# no-grad-context-mgr-update — ex2: implement companion EnableGrad context manager (re-enable inside NoGrad)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `no-grad-context-mgr-update`. Running the final beacon cell reports progress against the `Backprop: no_grad ctx-mgr update` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: no_grad ctx-mgr update` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-grad-context-mgr-update`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-grad-context-mgr-update"
DD_SUBTOPIC = "Backprop: no_grad ctx-mgr update"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `no_grad` / `enable_grad` context managers — quick refresher

Two complementary save-and-restore managers around the module-level `grad_tracking_enabled` flag:

```python
class NoGrad:      # disables grad inside the block
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
    def __exit__(self, *exc):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev

class EnableGrad:  # re-enables grad inside the block (cancels outer NoGrad)
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = True
    def __exit__(self, *exc):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev
```

Both save the PREVIOUS value (so nesting works) and restore on exit (even if the block raises). The only difference is the value they write on `__enter__`. `EnableGrad` is what you wrap a backward call in when you're inside `torch.no_grad()` but still need an inner gradient (e.g. higher-order grad inside an inference loop).

### Exercise 2 — implement companion EnableGrad context manager (re-enable inside NoGrad)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the save-and-restore context-manager pattern to write EnableGrad — the companion of NoGrad that re-enables grad_tracking_enabled inside its block and restores the previous value on exit, so it can cancel an outer NoGrad temporarily.
> Keywords: context-manager, enable-grad, nested, save-restore
> ```

**KCs targeted:** `no-grad-context-mgr-update`, `grad-tracking-global-toggle`

Implement `EnableGrad` — the companion context manager to `NoGrad`. It SETS `grad_tracking_enabled = True` on enter (the opposite write from `NoGrad`), saves whatever the flag was BEFORE, and restores it on exit:

```python
with NoGrad():
    # grad disabled
    with EnableGrad():
        # grad RE-ENABLED here (cancels outer NoGrad)
        ...
    # back to disabled (outer NoGrad's previous state)
```

**Use case.** You're inside a long inference path that disabled grad globally, but you need ONE inner block to compute a gradient (e.g. computing a Hessian-vector product for a small second-order optimizer step). `EnableGrad` cancels the outer `NoGrad` for the duration of its `with` block.

**Requirements.**
- `__enter__` saves the previous value of `grad_tracking_enabled` and sets it to `True`.
- `__exit__` restores the saved previous value — NOT hard-coded `False`.
- Works nested inside `NoGrad`. Works nested inside another `EnableGrad`. Works on its own.
- Restores on exit even if the block raised.

`grad_tracking_enabled` is a **module-level** name — use `global` to write to it. A `NoGrad` reference implementation is already in scope from the preamble for the nesting tests; if you want to use it directly, the test re-defines it as a fresh class so the two managers compose cleanly.

In [ ]:
class EnableGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled   # stash so nesting works
        grad_tracking_enabled = True
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev   # restore PREVIOUS value
        # implicit return None — exceptions propagate normally.


<details><summary>Solution</summary>

```python
class EnableGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled   # stash so nesting works
        grad_tracking_enabled = True
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev   # restore PREVIOUS value
        # implicit return None — exceptions propagate normally.
```

**Symmetry with NoGrad.** Same template, opposite write: NoGrad sets `False`, EnableGrad sets `True`. Both stash `self._prev` to support arbitrary nesting. This symmetry is the same one PyTorch exposes as `torch.no_grad()` and `torch.enable_grad()` — the latter exists precisely for the 'cancel an outer no_grad' case demonstrated in the test.

**Why no try/finally.** The `__exit__` method is called by the interpreter regardless of whether the body raised; explicit try/finally would be redundant. Returning `None` (or anything falsy) tells Python to propagate the exception.

**Cancelling vs over-riding.** The convention is that a child `EnableGrad` OVERRIDES its parent for the duration, then restores the parent's state. This matches what users expect from any `with`-based override (volume control, log level, verbosity flag) — the inner block is a temporary patch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()